# 13 – linker_v5 results (AIDev v5, agent follow-ups)
Reads `results/` from `step4_run_linker_v5.py` and answers:
1. How many merged AI PRs (Python, 500+ stars) have a later **agent** PR touching the same files, within 30 / 60 days? Per agent.
2. What kind of change is the follow-up (fix, refactor, feat…)? `all_changes` vs `fix_only`.
3. Same agent or another agent? How fast?
4. A sample of 50 pairs for the manual analysis.

> Candidate pairs only: same files + later date does **not** prove the AI PR was wrong (code-level analysis needed). Human follow-ups are **not** included (AIDev v5 has no human PRs).

In [ ]:
import os, sys, json
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "../..")
from common import bar, show_numbers, save_table, save_chart, stacked
from paths import RAW_V5, RESULTS
runs = {r: os.path.join(RESULTS, r) for r in ("all_changes_30_60", "fix_only_30_60")}
meta = {r: json.load(open(os.path.join(p, "run_metadata.json"))) for r, p in runs.items()}
for r, m in meta.items():
    print(f"{r}: {m['n_original_prs']:,} merged AI PRs | {m['n_candidate_pairs']:,} candidate pairs | horizon {m['observation_horizon']}")

## 1. AI PRs with a later agent change, per agent

In [ ]:
tabs = []
for r, p in runs.items():
    s = pd.read_csv(os.path.join(p, "per_agent.csv"))
    s = s[["agent", "n_prs", "pct_followup_30d_unc", "pct_followup_60d_unc"]].round(1)
    s.columns = ["agent", "merged AI PRs", f"% within 30 d ({r.split('_30')[0]})", f"% within 60 d ({r.split('_30')[0]})"]
    tabs.append(s.set_index(["agent", "merged AI PRs"]))
per_agent = pd.concat(tabs, axis=1).reset_index()
display(save_table(per_agent, "1_per_agent"))
print("(% over uncensored PRs: the whole window is inside the data)")

## 2. What kind of change is the follow-up? (all_changes)

In [ ]:
pairs = pd.read_parquet(os.path.join(runs["all_changes_30_60"], "candidate_pairs.parquet"))
pairs["followup_task_type"] = pairs["followup_task_type"].fillna("(none)")
bar(pairs.followup_task_type.value_counts(), "Type of the follow-up PR (all_changes, 60 days)", "2_followup_types.png", "pairs", pct=True)

## 3. Same agent or another agent? How fast?

In [ ]:
pairs["same_agent"] = (pairs.original_agent == pairs.followup_agent).map({True: "same agent", False: "other agent"})
display(save_table(pd.crosstab(pairs.original_agent, pairs.same_agent, margins=True, margins_name="TOTAL").reset_index(), "3_same_agent"))
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pairs.days_after_merge, bins=60, color="#1F4E79")
ax.set_xlabel("days after the AI PR was merged"); ax.set_ylabel("pairs"); ax.set_title("Time to the follow-up")
save_chart(fig, "3_days_to_followup.png")
display(pairs.days_after_merge.describe().round(1).to_frame("days"))

## 4. Pairs with links + sample for the manual analysis

In [ ]:
pr = pd.read_parquet(os.path.join(RAW_V5, "pull_request.parquet"), columns=["id", "title", "html_url"])
pr = pr.drop_duplicates("id").set_index("id")
pairs["ai_pr_url"] = pairs.original_pr_id.map(pr.html_url)
pairs["ai_pr_title"] = pairs.original_pr_id.map(pr.title)
pairs["followup_url"] = pairs.followup_pr_id.map(pr.html_url)
pairs["followup_title"] = pairs.followup_pr_id.map(pr.title)
pairs["shared_files"] = pairs.shared_files.map(lambda x: "; ".join(x))
cols = ["original_agent", "ai_pr_url", "ai_pr_title", "followup_agent", "followup_url", "followup_title",
        "followup_task_type", "days_after_merge", "within_30d", "shared_file_count", "shared_files"]
sample = pairs[pairs.within_30d].sample(min(50, int(pairs.within_30d.sum())), random_state=42)[cols].copy()
for c in ["why_changed (Q1)", "what_was_wrong (Q2)", "how_changed (Q3)", "is_real_fix (yes/no)", "notes"]:
    sample[c] = ""
display(sample.head(10))
with pd.ExcelWriter(os.path.join("tables", "linker_v5_results.xlsx")) as xw:
    per_agent.to_excel(xw, sheet_name="1 Per agent", index=False)
    pairs.followup_task_type.value_counts().rename_axis("type").reset_index().to_excel(xw, sheet_name="2 Follow-up types", index=False)
    pairs[cols].head(1_000_000).to_excel(xw, sheet_name="All pairs (all_changes)", index=False)  # Excel row limit
    sample.to_excel(xw, sheet_name="Sample to annotate (50)", index=False)
print("Saved tables/linker_v5_results.xlsx")